# Stage 06 - Create the Integrated Test Ontology

Bind the ontology package to the authoritative Stage 03 Lakehouse products and Stage 04 Eventhouse tables.

In [ ]:
import importlib.util
from pathlib import Path
import sys

if importlib.util.find_spec('fabricontology') is None:
    accelerator = Path('/lakehouse/default/Files/fabric-demos/06-ai-data-agent/Ontology/fabriciq_ontology_accelerator-0.1.0-py3-none-any.whl')
    if not accelerator.is_file():
        raise RuntimeError('Upload the bundled ontology accelerator with push-demo-06 before running this notebook.')
    sys.path.insert(0, str(accelerator))
from fabricontology import generate_definition_from_package
print('Ontology accelerator loaded from the Fabric Environment or bundled pure-Python wheel.')

In [ ]:
import json

import sempy.fabric as fabric
from fabricontology import generate_definition_from_package

workspace_id = fabric.get_workspace_id()
ontology_package_path = '/lakehouse/default/Files/fabric-demos/06-ai-data-agent/Ontology/mda_test_ontology.iq'
ontology_item_name = 'mda_test_ontology'
items_df = fabric.list_items()

def resolve_item(name, kind):
    matches = items_df[(items_df['Type'] == kind) & (items_df['Display Name'] == name)]
    if len(matches) != 1:
        raise ValueError(f'Expected exactly one {kind}: {name}')
    return str(matches.iloc[0].Id)

client = fabric.FabricRestClient()
database_id = resolve_item('kqldb_mda_test', 'KQLDatabase')
database = client.get(f'/v1/workspaces/{workspace_id}/kqlDatabases/{database_id}').json()
ontology_definition, entity_types, relationship_types, data_bindings, contextualizations = generate_definition_from_package(
    ontology_package_path=ontology_package_path,
    ontology_name=ontology_item_name,
    binding_workspace_id=workspace_id,
    binding_lakehouse_item_id=resolve_item('IntegratedTestLakehouse', 'Lakehouse'),
    binding_lakehouse_schema_name='',
    binding_eventhouse_item_id=resolve_item('eh_mda_test', 'Eventhouse'),
    binding_eventhouse_cluster_uri=database['properties']['queryServiceUri'],
    binding_eventhouse_database_name=database_id,
)
existing = items_df[(items_df['Type'] == 'Ontology') & (items_df['Display Name'] == ontology_item_name)]
if len(existing) > 1:
    raise ValueError('Multiple ontology items have the requested name')
if existing.empty:
    response = client.post(f'/v1/workspaces/{workspace_id}/ontologies', json={'displayName': ontology_item_name, 'definition': ontology_definition})
else:
    response = client.post(f'/v1/workspaces/{workspace_id}/ontologies/{existing.iloc[0].Id}/updateDefinition', json={'definition': ontology_definition})
response.raise_for_status()
print(json.dumps({'status': response.status_code, 'operation': response.headers.get('Location'), 'entities': len(entity_types), 'relationships': len(relationship_types)}))
print('Verify any asynchronous operation and graph refresh before claiming readiness.')